In [1]:
import pandas as pd
import os

# List of the exact files you want to combine
target_files = [
    "July hygiene.xlsx",
    "July soaps.xlsx",
    "August hygiene.xlsx",
    "August soaps.xlsx",
    "September hygiene.xlsx",
    "September soaps.xlsx",
    "October soaps.xlsx",
    "October hygiene.xlsx",
    "November soaps.xlsx",
    "November hygiene.xlsx",
    "December soaps.xlsx",
    "December hygiene.xlsx",
    "January soaps.xlsx",
    "January hygiene.xlsx",
    "February soaps.xlsx",
    "February hygiene.xlsx"

]

# Combine only those files that actually exist in the folder
combined_df = pd.concat(
    [
        pd.read_excel(f, sheet_name="DATA").assign(Source=os.path.splitext(f)[0])
        for f in target_files
        if os.path.exists(f)
    ],
    ignore_index=True
)

# Save the combined file
combined_df.to_excel("Combined_DATA.xlsx", index=False)

print("✅ Successfully combined selected Excel files into 'Combined_DATA.xlsx'")
print("📁 Files combined:", [f for f in target_files if os.path.exists(f)])
print("🧾 Total rows combined:", len(combined_df))


✅ Successfully combined selected Excel files into 'Combined_DATA.xlsx'
📁 Files combined: ['July hygiene.xlsx', 'July soaps.xlsx', 'August hygiene.xlsx', 'August soaps.xlsx', 'September hygiene.xlsx', 'September soaps.xlsx', 'October soaps.xlsx', 'October hygiene.xlsx', 'November soaps.xlsx', 'November hygiene.xlsx', 'December soaps.xlsx', 'December hygiene.xlsx', 'January soaps.xlsx', 'January hygiene.xlsx', 'February soaps.xlsx', 'February hygiene.xlsx']
🧾 Total rows combined: 145287


In [2]:
import pandas as pd

# Dictionary: Branch → Handler
branch_handlers = {
    "COVO-LAVINGTON": "Shamita Makena",
    "FRESHMARKET": "Cynthia Anyango",
    "GREEN PARK": "Anne Lole",
    "KAHAWA WEST": "Isabella Wambui",
    "KERUGOYA": "Hilda Murukira",
    "KIAMBU": "Juliet Nduta",
    "KIKUYU": "Dorcas Paul",
    "K-MALL": "Veronica Mwende",
    "LANGATA": "Dennis Kipruto",
    "LIMURU": "Dorcas Paul",
    "NAKURU": "Jane Njeri",
    "NGONG": "Joel Simiyu",
    "NYAHURURU": "Francis Murage",
    "QWETU RIVERSIDE": "Victor Mwaka",
    "RONGAI": "Irene Nthenya",
    "RUAKA": "Zakayo Vincent",
    "SOUTH B": "James Karugu",
    "WENDANI": "Linet Atieno"
}

# Load your combined file
df = pd.read_excel("Combined_DATA.xlsx")

# Create a new column 'Handler' by mapping Branch values
df["Handler"] = df["Branch"].map(branch_handlers)

# Save back to Excel (optional)
df.to_excel("Combined_DATA_with_Handlers.xlsx", index=False)

print("✅ Handlers successfully attached to each branch!")
print(df[["Branch", "Handler"]].head())


✅ Handlers successfully attached to each branch!
           Branch          Handler
0  COVO-LAVINGTON   Shamita Makena
1     FRESHMARKET  Cynthia Anyango
2          K-MALL  Veronica Mwende
3        KERUGOYA   Hilda Murukira
4          KIKUYU      Dorcas Paul


In [3]:
import pandas as pd

# Branch → Region mapping
branch_regions = {
    # Nairobi Region
    "COVO-LAVINGTON": "Nairobi Region",
    "FRESHMARKET": "Nairobi Region",
    "GREEN PARK": "Nairobi Region",
    "KAHAWA WEST": "Nairobi Region",
    "K-MALL": "Nairobi Region",
    "LANGATA": "Nairobi Region",
    "NGONG": "Nairobi Region",
    "QWETU RIVERSIDE": "Nairobi Region",
    "RONGAI": "Nairobi Region",
    "RUAKA": "Nairobi Region",
    "SOUTH B": "Nairobi Region",
    "WENDANI": "Nairobi Region",

    # Central Region
    "KIAMBU": "Central Region",
    "KIKUYU": "Central Region",
    "LIMURU": "Central Region",
    "KERUGOYA": "Central Region",

    # Rift Valley Region
    "NAKURU": "Rift Valley Region",
    "NYAHURURU": "Rift Valley Region"
}

# Load your combined dataset
df = pd.read_excel("Combined_DATA_with_Handlers.xlsx")

# Add Region column
df["Region"] = df["Branch"].map(branch_regions)

# Save the updated file
df.to_excel("Combined_DATA_with_Handlers_Regions.xlsx", index=False)

print("✅ Region and handler mapping completed successfully!")
print(df[["Branch", "Handler", "Region"]].head())

✅ Region and handler mapping completed successfully!
           Branch          Handler          Region
0  COVO-LAVINGTON   Shamita Makena  Nairobi Region
1     FRESHMARKET  Cynthia Anyango  Nairobi Region
2          K-MALL  Veronica Mwende  Nairobi Region
3        KERUGOYA   Hilda Murukira  Central Region
4          KIKUYU      Dorcas Paul  Central Region


In [4]:
import pandas as pd
import re

# Load your data
df = pd.read_excel("Combined_DATA_with_Handlers_Regions.xlsx")

def clean_class_name(row):
    cls = str(row["Class Name"]).strip().upper()
    desc = str(row["Product Desc"]).strip().upper()

    # === GLOBAL PRIORITY RULES (must run first) ===
    # Rush multipurpose → Multipurpose Dishwashing
    if "RUSH" in desc and ("M/PURP" in desc or "M-PURP" in desc or "PURPOSE" in desc):
        return "MULTIPURPOSE DISHWASHING"
    if re.search(r"ARIEL|SUNLIGHT|OMO|TOSS",desc):
        return "DETERGENTS"
    # Persil / Sunlight / Lavik handwash → Detergents
    if any(b in desc for b in ["PERSIL", "SUNLIGHT", "LAVIK", "ARIEL", "OMO",]) and \
       any(hw in desc for hw in ["HW", "HANDWASH", "HAND WASH", "H/WASH", "H/W"]):
        return "DETERGENTS"

    # Lavik disinfectant → Disinfectants
    if "LAVIK" in desc and "DISINFECT" in desc:
        return "DISINFECTANTS"

    # Topex value pack → Regular bleach
    if "TOPEX" in desc and "VALUE" in desc:
        return "REGULAR BLEACH"
    if "HURRICANE" in desc:
        return "TOILET CLEANERS"

    # Cerrazo cleaner → Home drycleaning agents
    if "CERRAZO" in desc:
        return "HOME DRYCLEANING AGENTS"

    # Magnee bathroom cleaner → Home drycleaning
    if "MAGNEE" in desc and "BATHROOM" in desc:
        return "BATHROOM CLEANERS"
    if "GLASS" in desc:
        return "GLASS"
    if "KITCHEN CLEANER" in desc and "ML" in desc:
        return "KITCHEN CLEANERS"
    if "STA-SOFT" in desc:
        return "FABRIC SOFTENERS"

    # === AIR FRESHENERS ===
    if cls == "AIR FRESHENERS":
        if re.search(r"T/?BALLS|MOTH BALLS|TOILET BALLS", desc):
            return "TOILET BALLS"
        elif re.search(r"T/?CLEANER|TOILET CLEANER", desc):
            return "TOILET CLEANERS"
        elif "BLOCK" in desc:
            return "TOILET BLOCKS"
        elif "CUDDLES" in desc:
            return "FABRIC SOFTENERS"
        elif "WINDOW" in desc or "GLASS" in desc:
            return "GLASS"

    # === DETERGENTS ===
    elif cls == "DETERGENTS":
        if "REG BLEACH" in desc or "REGULAR BLEACH" in desc:
            return "REGULAR BLEACH"
        if "PASTE" in desc:
            return "DISHWASHING PASTE"
        elif re.search(r"SCOURING POWDER|P/LIME", desc):
            return "SCOURING POWDERS"
        elif re.search(r"DISHWASH|DW|D/WASH|LIQUID DETERGENT|W/LIQUID|DISH WASHING|PRIDE LIQUID|SAFISHA LIQUID", desc):
            return "DISHWASHING"
        elif re.search(r"ALL PURPOSE|M-PURP|M/PURPOSE|ALLPURPOSE|MULTI PURPOSE", desc):
            return "MULTIPURPOSE DISHWASHING"
        elif re.search(r"BODY WASH|S/GEL|SHOWER GEL", desc):
            return "SHOWER GELS"
        elif re.search(r"HW|H/WASH|H/W|HANDWASH|HAND WASH|HAND LIQ", desc):
            return "HANDWASH AND SANITIZERS"
        elif re.search(r"STEELEX|SCRUB|STEEL|SHINER|BRIGHT|BRITE|SPONGE|PAD|PADS", desc):
            return "SCRUBBERS"
        elif re.search(r"BATHROOM", desc):
            return "BATHROOM CLEANERS"
        elif re.search(r"BODYWASH", desc):
            return "SHOWER GELS"

    # === DISINFECTANTS ===
    elif cls == "DISINFECTANTS":
        if re.search(r"BALLS?", desc):
            return "TOILET BALLS"
        elif "BLOCK" in desc:
            return "TOILET BLOCKS"
        elif "TOPEX" in desc:
            return "HOME DRYCLEANING AGENTS"
        elif re.search(r"DISHWASH|DISH WASHING|DWL", desc):
            return "DISHWASHING"
        elif re.search(r"AQUAGUARD|WATER GUARD", desc):
            return "WATER PURIFIERS"

    # === ANTISEPTICS ===
    elif cls == "ANTISEPTICS":
        if "HANDWASH" in desc:
            return "HANDWASH AND SANITIZERS"
        if "MAGNEE" in desc:
            return "HOME DRYCLEANING AGENTS"

    # === BATH / BAR SOAP ===
    elif cls in ["BATH SOAP", "BAR SOAPS"]:
        if re.search(r"BODY WASH|S/GEL|SHOWER GEL|B/WASH|BODYWASH", desc):
            return "SHOWER GELS"
        if re.search (r"LIQUID|LIQ|H/W|HANDWASH|HAND WASH",desc):
            return "HANDWASH AND SANITIZERS"
        elif "D/WASH" in desc:
            return "DISHWASHING"
        elif "CUDDLES" in desc:
            return "FABRIC SOFTENERS"
        
    # === BLEACHES ===
    elif cls == "BLEACHES":
        if "POWDER" in desc:
            return "DETERGENTS"
        if "SCOUR" in desc:
            return "SCOURING POWDERS"
        if re.search(r"COL|COLOUR|COLOR", desc):
            return "COLOURED BLEACH"
        elif "LIME" in desc or "DW" in desc:
            return "DISHWASHING"
        elif "PERSIL" in desc:
            return "DETERGENTS"
        elif "ACE" in desc:
            return "TOILET CLEANERS"
        elif "TABS" in desc:
            return "DETERGENTS"
        elif "ALLPUR" in desc:
            return "MULTIPURPOSE DISHWASHING"
        else:
            return "REGULAR BLEACH"

    # === HANDWASH & SANITIZERS CATEGORY ===
    elif "HANDWASH" in cls or "SANITIZER" in cls:
        if "SANITIZER" in desc:
            return "SPIRITS"
        if "WINDOW" in desc or "GLASS" in desc:
            return "GLASS"
        if re.search (r"B/WASH|BODY WASH|BODYWASH", desc):
            return "SHOWER GELS"
        if re.search (r"HAND SOAP|H/SOAP",desc):
            return "HANDWASH AND SANITIZERS"
        if "SOAP" in desc:
            return "BATH SOAP"
        if "LIME" in desc:
            return "HOME DRYCLEANING AGENTS"
        return "HANDWASH AND SANITIZERS"

    # === TOILET CLEANERS ===
    elif cls == "TOILET CLEANERS":
        if "CERRAZO" in desc:
            return "HOME DRYCLEANING AGENTS"
        if re.search (r"SCOURER|CANISTER",desc):
            return "SCOURING POWDERS"
        if "LAVIK" in desc:
            return "DISINFECTANTS"
        if "PRIDE POWDER" in desc:
            return "SCOURING POWDERS"
        if "PRIDE LQ" in desc:
            return "DISHWASHING"
        if "BLUE" in desc and "BUBBLE" in desc:
            return "TOILET BLOCKS"
        if "PRIDE M" in desc:
            return "MULTIPURPOSE DISHWASHING"
        if "WINDOW" in desc or "GLASS" in desc:
            return "GLASS"
        if "BATHROOM" in desc:
            return "HOME DRYCLEANING AGENTS"
        if "BLOCK" in desc:
            return "TOILET BLOCKS"
        if "BALL" in desc:
            return "TOILET BALLS"
        if "HANDWASH" in desc:
            return "HANDWASH AND SANITIZERS"
        if "TOPEX" in desc:
            return "HOME DRYCLEANING AGENTS"
    elif cls == "SCOURING AGENTS-S/BRITES,S/WOO":
        if "TOILET CLEANER" in desc:
            return "TOILET CLEANERS"
        elif re.search(r"DOMESTOS POWDER|SCOURING PWD|SCOURING POWDER|SCOURING POWER|VIM SCOURER|CANNISTER|POUCH|VIM ALL-PURPOSE|LEMON FRESH SCOURING", desc):
            return "SCOURING POWDERS"
        if "PASTE" in desc:
            return "DISHWASHING PASTE"
        if "BATHROOM" in desc and "BATHROOM" in desc:
            return "BATHROOM CLEANERS"
        if "SAFISHA W/C" in desc:
            return "TOILET CLEANERS"
        if "PRIDE M-PURP" in desc:
            return "MULTIPURPOSE DISHWASHING"
        elif re.search(r"D/W|DISHWASH|D/WASHING|DISHWASHING|DW|W/LIQUID|DWL|LIQUID", desc):
            return "DISHWASHING"
    elif cls == "SCOURING POWDERS":
        if "WASHING POWDER" in desc:
            return "DETERGENTS"
    elif cls == "HOME DRYCLEANING AGENTS":
        if "HAND" in desc:
            return "HANDWASH AND SANITIZERS"
    
        
    return cls.replace("&", "AND").replace("SANiTIZERS", "SANITIZERS")

df["Cleaned Class Name"] = df.apply(clean_class_name, axis=1)
df.to_excel("Cleaned_data.xlsx", index=False)

print("✅ Cleaning completed successfully!")


✅ Cleaning completed successfully!
